In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, desc, sum, when

spark = (
    SparkSession.builder.appName("TugasMandiriPertemuan4")
    .master("local[2]")
    .config("spark.driver.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Baca dataset
df = spark.read.csv(
    "transaksi_september_2026.csv", header=True, inferSchema=True
)

In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [5]:
# 1. Baca dataset
df = spark.read.csv("transaksi_september_2026.csv", header=True, inferSchema=True)

# 2. Tampilkan printSchema()
print("SCHEMA DATASET")
df.printSchema()

# 3. Tampilkan jumlah baris
print(f"Jumlah baris: {df.count()}")

# 4. Tampilkan 10 baris pertama
print("\n10 BARIS PERTAMA")
df.show(10)

SCHEMA DATASET
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000

10 BARIS PERTAMA
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehat

In [9]:
# Tangani data kosong pada kolom rating
df_clean = df.na.fill({"rating": 0})

# Menangani nilai kosong dengan mengisi nilai rata-rata (misal: 4.0) atau mengisi nilai default
df_clean = df.na.fill({"rating": 0})
print("Data kosong berhasil ditangani.")

Data kosong berhasil ditangani.


In [10]:
# Membuat variabel df_transformed
df_transformed = df_clean.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga_satuan")
).withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"),
)

# Cek hasil
df_transformed.show(5)# 1 Kategori yang memiliki pendapatan tertinggi
df_transformed.groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(desc("total_pendapatan")) \
    .show(1)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [11]:
# 1 Kategori yang memiliki pendapatan tertinggi
df_transformed.groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(desc("total_pendapatan")) \
    .show(1)

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row



In [12]:
#2 kota dengan jumlah transaksi besar terbanyak
df_transformed.filter(col("tier_transaksi") == "Besar") \
              .groupBy("kota") \
              .count() \
              .orderBy(desc("count")) \
              .show(1)

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row



In [13]:
#3 rata-rata untuk masing-masing pembayaran
df_transformed.groupBy("metode_pembayaran") \
              .agg(avg("rating").alias("rata_rata_rating")) \
              .show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [14]:
#E. simpan file
df_transformed.write.mode("overwrite").csv("output_tugas4_september", header=True)
print("Berhasil menyimpan data hasil olahan!")

Berhasil menyimpan data hasil olahan!
